In [1]:
!pip install torch torchvision matplotlib -q

In [2]:
"""
Self-Pruning Neural Network on CIFAR-10
========================================
Tredence AI Engineering Internship — Case Study Solution

Problem: Design a neural network that learns to prune its own weights
during training using learnable gate parameters and L1 sparsity loss.

Total Loss = CrossEntropyLoss + λ × SparsityLoss

Author: [Your Name]
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os
import time


# ══════════════════════════════════════════════════════
# PART 1: PrunableLinear — Custom Gated Linear Layer
# ══════════════════════════════════════════════════════

class PrunableLinear(nn.Module):
    """
    A drop-in replacement for nn.Linear that learns which weights to prune.

    Each weight w_ij has a corresponding learnable scalar gate_score g_ij.
    During the forward pass:

        gates       = sigmoid(gate_scores)     →  values strictly in (0, 1)
        pruned_w    = weight  ×  gates         →  element-wise soft masking
        output      = F.linear(x, pruned_w, bias)

    When the optimizer drives gate_scores to very negative values,
    sigmoid pushes the gate to ≈ 0, effectively removing that weight
    from the network — this is the "pruning" effect.

    Gradient flow (verified by autograd):
        ∂L/∂weight      flows through pruned_w  →  F.linear   ✓
        ∂L/∂gate_scores flows through sigmoid   →  pruned_w   ✓

    Parameters
    ----------
    in_features  : int
    out_features : int
    """

    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features

        # ── Standard parameters (same as nn.Linear) ──────────────────────
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias   = nn.Parameter(torch.zeros(out_features))

        # ── Gate scores — one per weight element ─────────────────────────
        # Initialized to -2.0 so that sigmoid(-2) ≈ 0.12.
        # Starting near 0 rather than 0.5 gives the L1 penalty a head-start:
        # gates already lean toward zero, making pruning easier to achieve.
        self.gate_scores = nn.Parameter(
            torch.full((out_features, in_features), -2.0)
        )

        # Kaiming init for weights — good default for ReLU activations
        nn.init.kaiming_uniform_(self.weight, a=0.01)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass: multiply each weight by its soft gate before the
        linear transform.  Both weight and gate_scores receive gradients.
        """
        gates = torch.sigmoid(self.gate_scores)        # ∈ (0, 1)
        return F.linear(x, self.weight * gates, self.bias)

    def get_gates(self) -> torch.Tensor:
        """Return current gate values, detached (for evaluation / plotting)."""
        return torch.sigmoid(self.gate_scores).detach()

    def sparsity_loss(self) -> torch.Tensor:
        """
        Mean gate value for this layer.
        Using the MEAN (not sum) keeps the loss scale-independent of layer
        size, so the same λ values work regardless of how wide the network is.
        """
        return torch.sigmoid(self.gate_scores).mean()


# ══════════════════════════════════════════════════════
# Network Definition
# ══════════════════════════════════════════════════════

class SelfPruningNet(nn.Module):
    """
    Feed-forward network for CIFAR-10 using PrunableLinear layers.

    Architecture:
        Input  (3×32×32 = 3072)
          → PrunableLinear(3072 → 1024) → ReLU
          → PrunableLinear(1024 → 512)  → ReLU
          → PrunableLinear(512  → 256)  → ReLU
          → PrunableLinear(256  → 10)   → logits
    """

    def __init__(self):
        super().__init__()
        self.fc1 = PrunableLinear(3072, 1024)
        self.fc2 = PrunableLinear(1024, 512)
        self.fc3 = PrunableLinear(512,  256)
        self.fc4 = PrunableLinear(256,  10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(x.size(0), -1)          # flatten: (B, 3072)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        return self.fc4(x)                  # raw logits

    # ── Sparsity helpers ──────────────────────────────────────────────────

    def sparsity_loss(self) -> torch.Tensor:
        """
        Mean gate value across ALL four PrunableLinear layers.
        This is the SparsityLoss term added to the total loss:

            Total Loss = CrossEntropyLoss + λ × sparsity_loss()

        Minimising this term pushes all gate values toward 0 (pruning).
        The mean normalisation keeps the value in [0, 1] regardless of
        network size, so λ is directly interpretable.
        """
        return (
            self.fc1.sparsity_loss() +
            self.fc2.sparsity_loss() +
            self.fc3.sparsity_loss() +
            self.fc4.sparsity_loss()
        ) / 4

    def sparsity_level(self, threshold: float = 0.05) -> float:
        """
        Fraction of weights whose gate < threshold (i.e. effectively pruned).
        Returns a value in [0, 1]; higher = sparser network.
        """
        gates = self.all_gate_values()
        return (gates < threshold).float().mean().item()

    def all_gate_values(self) -> torch.Tensor:
        """Flat tensor of all gate values across every layer (for plotting)."""
        return torch.cat([
            self.fc1.get_gates().view(-1),
            self.fc2.get_gates().view(-1),
            self.fc3.get_gates().view(-1),
            self.fc4.get_gates().view(-1),
        ])


# ══════════════════════════════════════════════════════
# Data Loading
# ══════════════════════════════════════════════════════

def get_data(batch_size: int = 128):
    """
    Download CIFAR-10 and return (train_loader, test_loader).
    No heavy augmentation — keeps training fast on CPU.
    """
    transform = transforms.Compose([transforms.ToTensor()])

    train_set = datasets.CIFAR10('./data', train=True,  download=True, transform=transform)
    test_set  = datasets.CIFAR10('./data', train=False, download=True, transform=transform)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(test_set,  batch_size=256,        shuffle=False)

    return train_loader, test_loader


# ══════════════════════════════════════════════════════
# PART 3: Training Loop
# ══════════════════════════════════════════════════════

@torch.no_grad()
def evaluate(model: SelfPruningNet, loader: DataLoader, device: torch.device) -> float:
    """Return test accuracy as a percentage."""
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total   += y.size(0)
    return 100.0 * correct / total


def train_model(
    lam: float,
    train_loader: DataLoader,
    test_loader: DataLoader,
    device: torch.device,
    epochs: int = 20,
) -> tuple:
    """
    Train one SelfPruningNet for `epochs` epochs with a given λ.

    Loss per batch:
        cls_loss  = CrossEntropyLoss(logits, labels)
        spar_loss = mean gate value across all layers   ∈ (0, 1)
        total     = cls_loss + λ × spar_loss

    A higher λ applies stronger pressure to push gates → 0.

    Returns
    -------
    model, final_test_accuracy (%), final_sparsity_level (%)
    """
    model     = SelfPruningNet().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    print(f"\n{'='*50}")
    print(f"  Training with λ = {lam}")
    print(f"{'='*50}")

    t0 = time.time()

    for epoch in range(1, epochs + 1):
        model.train()

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()

            logits    = model(x)
            cls_loss  = criterion(logits, y)          # classification loss
            spar_loss = model.sparsity_loss()          # L1 gate penalty (normalised mean)
            loss      = cls_loss + lam * spar_loss     # combined loss

            loss.backward()   # gradients flow to both weight & gate_scores
            optimizer.step()

        # ── Progress report every 5 epochs ──
        if epoch % 5 == 0 or epoch == 1:
            acc      = evaluate(model, test_loader, device)
            sparsity = model.sparsity_level() * 100
            print(
                f"  Epoch {epoch:2d}/{epochs} | "
                f"Acc: {acc:.2f}% | "
                f"Sparsity: {sparsity:.2f}% | "
                f"Time: {time.time() - t0:.1f}s"
            )

    final_acc      = evaluate(model, test_loader, device)
    final_sparsity = model.sparsity_level() * 100

    print(f"\n  ✓ Final Accuracy : {final_acc:.2f}%")
    print(f"  ✓ Final Sparsity : {final_sparsity:.2f}%")

    return model, final_acc, final_sparsity


# ══════════════════════════════════════════════════════
# Plotting
# ══════════════════════════════════════════════════════

def plot_gate_distribution(model: SelfPruningNet, lam: float, path: str):
    """
    Histogram of all gate values.

    A SUCCESSFUL result shows:
      • Large spike at 0   → many pruned (dead) connections
      • Cluster near 1     → surviving important connections
      • Very few values in the middle (gates converge to binary states)
    """
    gates = model.all_gate_values().cpu().numpy()
    sparsity_pct = (gates < 0.05).mean() * 100

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(
        f"Gate Value Distribution  (λ = {lam})  |  "
        f"Sparsity: {sparsity_pct:.1f}%  |  Total gates: {len(gates):,}",
        fontsize=12, fontweight="bold"
    )

    # Full distribution (log scale to see both spike and tail)
    axes[0].hist(gates, bins=100, color="#2563EB", edgecolor="white", linewidth=0.3)
    axes[0].set_yscale("log")
    axes[0].axvline(0.05, color="red", linestyle="--", linewidth=1.5,
                    label="Prune threshold (0.05)")
    axes[0].set_xlabel("Gate Value")
    axes[0].set_ylabel("Count (log scale)")
    axes[0].set_title("Full Distribution")
    axes[0].legend()

    # Zoom near 0 to show the pruning spike clearly
    near_zero = gates[gates < 0.15]
    if len(near_zero) > 0:
        axes[1].hist(near_zero, bins=60, color="#DC2626", edgecolor="white", linewidth=0.3)
        axes[1].axvline(0.05, color="black", linestyle="--", linewidth=1.5,
                        label="Prune threshold")
        axes[1].set_xlabel("Gate Value  (zoomed: 0 – 0.15)")
        axes[1].set_ylabel("Count")
        axes[1].set_title("Zoom: Near-Zero (Pruned) Gates")
        axes[1].legend()
    else:
        axes[1].text(0.5, 0.5, "No near-zero gates found",
                     ha="center", va="center", transform=axes[1].transAxes)

    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Plot saved → {path}")


def plot_tradeoff(results: list, path: str):
    """Dual-axis plot: accuracy and sparsity vs lambda."""
    lambdas = [r["lambda"]   for r in results]
    accs    = [r["accuracy"] for r in results]
    spars   = [r["sparsity"] for r in results]

    fig, ax1 = plt.subplots(figsize=(8, 5))

    ax1.set_xlabel("Lambda (λ)", fontsize=12)
    ax1.set_ylabel("Test Accuracy (%)", color="#2563EB", fontsize=12)
    ax1.plot(lambdas, accs, "o-", color="#2563EB", linewidth=2,
             markersize=8, label="Accuracy")
    ax1.tick_params(axis="y", labelcolor="#2563EB")

    ax2 = ax1.twinx()
    ax2.set_ylabel("Sparsity Level (%)", color="#DC2626", fontsize=12)
    ax2.plot(lambdas, spars, "s--", color="#DC2626", linewidth=2,
             markersize=8, label="Sparsity")
    ax2.tick_params(axis="y", labelcolor="#DC2626")

    lines  = ax1.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
    labels = ax1.get_legend_handles_labels()[1] + ax2.get_legend_handles_labels()[1]
    ax1.legend(lines, labels, loc="center right")

    plt.title("Accuracy–Sparsity Trade-off vs λ", fontsize=13, fontweight="bold")
    fig.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Trade-off plot saved → {path}")


# ══════════════════════════════════════════════════════
# Main Experiment
# ══════════════════════════════════════════════════════

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n🔧 Device: {device}")

    torch.manual_seed(42)
    np.random.seed(42)

    os.makedirs("results", exist_ok=True)

    print("\n📦 Loading CIFAR-10 …")
    train_loader, test_loader = get_data(batch_size=128)

    # ── Three λ values: low / medium / high ──────────────────────────────
    # Because sparsity_loss is normalised (mean gate value ∈ [0,1]):
    #   λ =  5  → mild pressure   → high accuracy, moderate sparsity
    #   λ = 10  → balanced        → medium accuracy, high sparsity
    #   λ = 20  → strong pressure → lower accuracy, very high sparsity
    lambda_values = [5, 10, 20]
    results       = []

    for lam in lambda_values:
        model, acc, sparsity = train_model(lam, train_loader, test_loader, device)
        results.append({"lambda": lam, "accuracy": acc, "sparsity": sparsity})
        plot_gate_distribution(model, lam, f"results/gates_lambda_{lam}.png")

    plot_tradeoff(results, "results/tradeoff.png")

    # ── Final summary table ───────────────────────────────────────────────
    print("\n" + "="*52)
    print(f"  {'Lambda':<10} {'Test Accuracy':<18} {'Sparsity Level'}")
    print("  " + "-"*46)
    for r in results:
        print(f"  {r['lambda']:<10} {r['accuracy']:<18.2f} {r['sparsity']:.2f}%")
    print("="*52)


if __name__ == "__main__":
    main()



🔧 Device: cpu

📦 Loading CIFAR-10 …


100%|██████████| 170M/170M [00:01<00:00, 86.7MB/s]



  Training with λ = 5
  Epoch  1/20 | Acc: 26.46% | Sparsity: 0.00% | Time: 73.3s
  Epoch  5/20 | Acc: 39.27% | Sparsity: 55.39% | Time: 399.2s
  Epoch 10/20 | Acc: 44.84% | Sparsity: 56.18% | Time: 813.7s
  Epoch 15/20 | Acc: 48.27% | Sparsity: 57.35% | Time: 1236.8s
  Epoch 20/20 | Acc: 49.90% | Sparsity: 58.75% | Time: 1644.1s

  ✓ Final Accuracy : 49.90%
  ✓ Final Sparsity : 58.75%
  Plot saved → results/gates_lambda_5.png

  Training with λ = 10
  Epoch  1/20 | Acc: 22.56% | Sparsity: 0.00% | Time: 71.0s
  Epoch  5/20 | Acc: 36.62% | Sparsity: 55.58% | Time: 414.6s
  Epoch 10/20 | Acc: 43.44% | Sparsity: 59.57% | Time: 842.4s
  Epoch 15/20 | Acc: 46.69% | Sparsity: 63.26% | Time: 1266.9s
  Epoch 20/20 | Acc: 49.45% | Sparsity: 66.42% | Time: 1706.2s

  ✓ Final Accuracy : 49.45%
  ✓ Final Sparsity : 66.42%
  Plot saved → results/gates_lambda_10.png

  Training with λ = 20
  Epoch  1/20 | Acc: 25.17% | Sparsity: 0.00% | Time: 77.0s
  Epoch  5/20 | Acc: 34.55% | Sparsity: 60.74% | T